# O2 A Core Primitive Bottleneck Demo

This notebook isolates the per-spectrum bottleneck by following the actual forward-model work from nominal O2 A samples to the LABOS matrix primitives. The goal is not only to show that a kernel is called many times. The goal is to explain why the code takes those steps, why the counts have their current values, and why individually cheap operations become expensive in aggregate.

Code sites:

- `src/forward_model/instrument_grid/grid_calculation/simulate.zig`
- `src/forward_model/instrument_grid/grid_calculation/wavelength_sampling.zig`
- `src/forward_model/implementations/instrument/adaptive_plan.zig`
- `src/forward_model/radiative_transfer/labos/execute.zig`
- `src/forward_model/radiative_transfer/labos/layers.zig`
- `src/forward_model/radiative_transfer/labos/matrix.zig`
- `tests/perf/labos_kernel_bench.zig`

In [ ]:
import re
import subprocess
from dataclasses import dataclass
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "build.zig").exists() and (path / "src").is_dir():
            return path
    raise RuntimeError("Run this notebook from inside the zdisamar repository")


REPO_ROOT = find_repo_root(Path.cwd())


@dataclass(frozen=True)
class KernelTiming:
    name: str
    ns_per_call: float


@dataclass(frozen=True)
class PrimitiveCall:
    label: str
    kernel: str
    calls: int

    def milliseconds(self, timings: dict[str, KernelTiming]) -> float:
        return self.calls * timings[self.kernel].ns_per_call / 1.0e6

## What the Per-Spectrum Path Does

One O2 A spectrum is requested on a nominal output grid, but the radiance is not evaluated only at those nominal wavelengths. The measurement is an instrument-convolved quantity. For DISAMAR parity, zdisamar first evaluates exact radiance at high-resolution support wavelengths, then integrates those exact values back to the nominal grid.

The core path is:

```text
701 nominal wavelengths
  -> build an instrument-response integration kernel for each nominal wavelength
  -> collect the unique exact radiance support wavelengths
  -> run the exact forward model once for each unique support wavelength
  -> integrate exact radiance values back to the 701 nominal samples
```

That high-resolution support step exists because O2 A absorption and solar structure vary sharply inside a single instrument channel. Evaluating only the channel center would erase the sub-channel structure that the instrument response is supposed to integrate.

## Why 701 Nominal Samples Becomes 3,874 Exact Wavelengths

The nominal grid count is direct input configuration: `start_nm=755.0`, `end_nm=776.0`, `sample_count=701`. That gives a 0.03 nm nominal spacing.

The exact support count is not `701 * some fixed kernel size`. The DISAMAR high-resolution response builds a global support plan over the band plus a response margin. O2 line centers split that plan into intervals, and each interval receives a Gauss quadrature order. The unique support count is the sum of those interval orders.

The important rules are:

```text
global_start = start_nm - 2 * fwhm_nm
global_end   = end_nm   + 2 * fwhm_nm
line threshold = max(line_strength) * threshold_line_sim
intervals are split at strong O2 line centers
interval divisions are clamped between strong_line_min_divisions and strong_line_max_divisions
```

For the O2 A reference case, this becomes:

In [ ]:
NOMINAL_GRID = {
    "start_nm": 755.0,
    "end_nm": 776.0,
    "sample_count": 701,
}

INSTRUMENT_RESPONSE = {
    "fwhm_nm": 0.38,
    "high_resolution_half_span_nm": 1.14,
}

STRONG_LINE_PLAN = {
    "threshold_line_sim": 3.0e-5,
    "strong_line_centers_in_window": 339,
    "support_intervals": 360,
    "division_histogram": {
        8: 272,
        9: 15,
        10: 10,
        11: 18,
        12: 10,
        13: 3,
        14: 1,
        16: 1,
        17: 1,
        19: 1,
        21: 1,
        22: 1,
        26: 1,
        27: 1,
        33: 1,
        34: 1,
        37: 1,
        40: 21,
    },
}

nominal_step_nm = (
    NOMINAL_GRID["end_nm"] - NOMINAL_GRID["start_nm"]
) / (NOMINAL_GRID["sample_count"] - 1)
global_start_nm = NOMINAL_GRID["start_nm"] - 2.0 * INSTRUMENT_RESPONSE["fwhm_nm"]
global_end_nm = NOMINAL_GRID["end_nm"] + 2.0 * INSTRUMENT_RESPONSE["fwhm_nm"]
exact_support_count = sum(
    division * interval_count
    for division, interval_count in STRONG_LINE_PLAN["division_histogram"].items()
)

print("Nominal grid")
print(f"  start/end nm               {NOMINAL_GRID['start_nm']:.3f} -> {NOMINAL_GRID['end_nm']:.3f}")
print(f"  nominal samples            {NOMINAL_GRID['sample_count']:>10,d}")
print(f"  nominal spacing nm         {nominal_step_nm:>10.6f}")
print()
print("DISAMAR high-resolution support plan")
print(f"  response FWHM nm           {INSTRUMENT_RESPONSE['fwhm_nm']:>10.3f}")
print(f"  global support window nm   {global_start_nm:.3f} -> {global_end_nm:.3f}")
print(f"  strong O2 centers          {STRONG_LINE_PLAN['strong_line_centers_in_window']:>10,d}")
print(f"  support intervals          {STRONG_LINE_PLAN['support_intervals']:>10,d}")
print(f"  exact support wavelengths  {exact_support_count:>10,d}")
print()
print("Division histogram: intervals at each Gauss order")
for division, interval_count in sorted(STRONG_LINE_PLAN["division_histogram"].items()):
    support_nodes = division * interval_count
    print(f"  order {division:2d}: {interval_count:3d} intervals -> {support_nodes:4d} wavelengths")

The number is therefore explainable:

```text
272 intervals at order 8   -> 2176 support wavelengths
21 intervals at order 40   ->  840 support wavelengths
remaining intervals        ->  858 support wavelengths
                           = 3874 exact wavelengths
```

Most intervals are cheap order-8 intervals. A smaller number of wider intervals gets higher order, up to order 40. The count is still large because the O2 line list puts hundreds of line centers inside the support window, and those line centers split the response plan before quadrature is assigned.

The deduplication policy in `wavelength_sampling.zig` is exact-bit deduplication of `radiance_wavelength_nm + integration.offset_nm`. It removes repeated support wavelengths, but it does not coarsen nearby wavelengths. Nearby wavelengths remain distinct exact radiative-transfer solves.

## Measured O2 A Work Counts

These counts are the per-spectrum core workload after support deduplication:

```text
701 nominal wavelengths -> 3874 exact radiance solves
3874 solves -> 120390 Fourier terms
RT-layer construction -> 5417550 layer visits
RT-layer doubling -> 8389666 doubling steps
```

The forward model spends little time integrating back to 701 nominal values. The expensive section is the exact-wavelength prefetch: each unique support wavelength runs configured input construction and LABOS transport.

In [ ]:
SPECTRUM_COUNTS = {
    "nominal_samples": 701,
    "exact_radiance_solves": 3874,
    "fourier_terms": 120390,
    "labos_layers": 5417550,
    "doubled_layers": 1075939,
    "double_steps": 8389666,
}

MATRIX_CALLS = [
    PrimitiveCall("Q = qseries(R * R)", "qseries_12x10", 3_408_299),
    PrimitiveCall("D = T + Q * diag(E) + Q * T", "smulAddSemul3_12", 3_408_299),
    PrimitiveCall("rd = R * D", "smul_12x10", 8_389_666),
    PrimitiveCall("U = R * diag(E) + rd", "semulAdd_12", 8_389_666),
    PrimitiveCall("tu = T * U", "smul_12x10", 8_389_666),
    PrimitiveCall("R_next = R + diag(E) * U + tu", "matAddEsmul3_12", 8_389_666),
    PrimitiveCall("td = T * D", "smul_12x10", 8_389_666),
    PrimitiveCall("T_next = diag(E) * D + T * diag(E) + td", "esmulSemulAdd_12", 8_389_666),
]

print("O2 A support fan-out")
for key, value in SPECTRUM_COUNTS.items():
    print(f"  {key:24s} {value:>10,d}")

fourier_terms_per_solve = (
    SPECTRUM_COUNTS["fourier_terms"] / SPECTRUM_COUNTS["exact_radiance_solves"]
)
labos_layers_per_fourier = (
    SPECTRUM_COUNTS["labos_layers"] / SPECTRUM_COUNTS["fourier_terms"]
)
double_steps_per_doubled_layer = (
    SPECTRUM_COUNTS["double_steps"] / SPECTRUM_COUNTS["doubled_layers"]
)
print(f"  Fourier terms / solve     {fourier_terms_per_solve:10.3f}")
print(f"  LABOS layers / Fourier    {labos_layers_per_fourier:10.3f}")
print(f"  double steps / doubled    {double_steps_per_doubled_layer:10.3f}")

## Why So Many Fourier Terms

LABOS expands the azimuthal dependence of the scattering field into Fourier components. The loop in `execute.zig` runs one transport solve per Fourier index:

```text
for m in 0..fourier_max:
    build PLM basis for m
    build RT layers for m
    solve multiple-scattering orders for m
    add weighted reflectance contribution for m
```

The O2 A case is not a near-normal geometry, so it cannot collapse to the scalar Fourier term. It uses `solar_zenith_deg=60`, `viewing_zenith_deg=30`, and `relative_azimuth_deg=120`. The aerosol phase function has `g=0.7`, so scattering is angularly structured enough that many azimuthal modes remain nonzero.

The measured average is `120390 / 3874 = 31.076` Fourier terms per exact wavelength. The non-integer average is expected: the Fourier tail is stopped per wavelength when the contribution falls below `3.0e-14` after the configured floor. Wavelength-dependent optical depth changes make the per-wavelength Fourier workload vary slightly.

In [ ]:
base_fourier_terms = SPECTRUM_COUNTS["fourier_terms"] // SPECTRUM_COUNTS["exact_radiance_solves"]
extra_fourier_terms = SPECTRUM_COUNTS["fourier_terms"] % SPECTRUM_COUNTS["exact_radiance_solves"]

print("Fourier count interpretation")
print(f"  base terms per exact solve       {base_fourier_terms:>10,d}")
print(f"  terms beyond 31/solve      {extra_fourier_terms:>10,d}")
print(f"  exact solves                     {SPECTRUM_COUNTS['exact_radiance_solves']:>10,d}")
print()
print(
    "The total is 31 terms per exact solve plus "
    f"{extra_fourier_terms:,} additional Fourier terms spread across the "
    "wavelength-local tail decisions."
)

## Why RT-Layer Construction Dominates

For each Fourier term, `calcRTlayersIntoWithBasis` visits the layer stack and builds a reflection/transmission operator for every active layer. That step is necessary because each Fourier component sees a different phase-kernel projection (`Zplus`, `Zmin`), and each exact wavelength has different optical depths from O2, O2-O2, Rayleigh, and aerosol contributions.

The doubling branch is triggered when the layer is too optically thick for a single-scattering layer approximation:

```text
use doubling when: a_eff * optical_depth > threshold_doubl
split until:       a_eff * (optical_depth / 2^ndouble) < threshold_doubl
```

For this case `threshold_doubl = 1.0e-6`. That is a strict threshold, so many active scattering layers are split into very thin starting layers and then doubled back up. The strict threshold keeps parity with the reference transport behavior, but it creates millions of repeated small matrix updates.

In [ ]:
DOUBLING_COUNTS = {
    "qseries_calls": 3_408_299,
    "double_steps": 8_389_666,
    "smul_calls": 3 * 8_389_666,
    "diag_add_calls": 4 * 8_389_666 + 3_408_299,
}

print("Doubling repetition pressure")
for key, value in DOUBLING_COUNTS.items():
    print(f"  {key:18s} {value:>12,d}")
print()
print(
    "qseries is not called on every doubling step because the trace gate can "
    "skip the Q-series path when the product is effectively zero."
)

## Run the Zig Primitive Benchmark

The benchmark uses the actual Zig kernels from `matrix.zig`. It is intentionally narrow: it measures the small matrix primitives, then the notebook multiplies those primitive timings by the O2 A call counts.

This keeps the demo focused on the LABOS wall by measuring the same primitive kernels used by the forward model.

In [ ]:
BENCH_RE = re.compile(r"^(?P<name>[a-zA-Z0-9_]+): .* ns_per_call=(?P<ns>[0-9.]+)")


def run_zig_bench() -> dict[str, KernelTiming]:
    completed = subprocess.run(
        ["zig", "build", "bench"],
        cwd=REPO_ROOT,
        check=True,
        capture_output=True,
        text=True,
    )
    output = completed.stdout + completed.stderr
    timings: dict[str, KernelTiming] = {}
    for line in output.splitlines():
        match = BENCH_RE.match(line.strip())
        if not match:
            continue
        name = match.group("name")
        timings[name] = KernelTiming(
            name=name,
            ns_per_call=float(match.group("ns")),
        )
    return timings


timings = run_zig_bench()
for timing in timings.values():
    print(f"{timing.name:28s} {timing.ns_per_call:10.3f} ns/call")

## Why Some Primitives Are Cheap and One Is Relatively Expensive

Most primitive calls are cheap because the matrix shape is fixed and small. The O2 A case uses `n_streams=20`, so `n_gauss=10`; with the direct solar and view directions, the LABOS matrix size is `n=12`. `matrix.zig` has specialized 12x10 and 12x12 paths for this shape. The arrays fit in cache, there is no heap allocation, and the innermost products are straight-line f64 multiply-add chains.

`smul_12x10` is a compact fixed-shape product. The diagonal add/update helpers are cheaper because they mostly scale or add 12x12 arrays.

`qseries_12x10` is more expensive because it does more than multiply two matrices. It forms `R * R`, checks the trace, builds `(I - R*R)` on the 10-stream Gauss block, performs pivoted factorization, solves for an inverse, and then applies the inverse back to the extra direct/view rows and columns. That means divisions, pivots, triangular solves, and dependent f64 operations. It is still sub-microsecond, but it is the most expensive primitive in the doubling update.

## Reconstruct the Doubling Primitive Cost

The doubling loop in `layers.zig` applies the same small set of matrix primitives millions of times. This cell multiplies isolated kernel timings by the measured O2 A call counts.

In [ ]:
rows = []
total_ms = 0.0
for primitive in MATRIX_CALLS:
    elapsed_ms = primitive.milliseconds(timings)
    total_ms += elapsed_ms
    rows.append((primitive.label, primitive.kernel, primitive.calls, elapsed_ms))

print("Primitive contribution estimate")
print(f"{'operation':46s} {'kernel':24s} {'calls':>12s} {'worker_ms':>12s}")
for label, kernel, calls, elapsed_ms in rows:
    print(f"{label:46s} {kernel:24s} {calls:12,d} {elapsed_ms:12.3f}")
print(f"{'total matrix primitive estimate':72s} {total_ms:12.3f}")

## Interpretation

The reconstruction should land in the same band as the measured doubling bucket. The exact number varies with CPU load and compiler output, but the result demonstrates the wall:

```text
millions of 12x10 / 12x12 f64 kernels
  * under 3874 exact radiance solves
  * under 120390 Fourier terms
  * under 8389666 doubling steps
```

Each primitive is cheap in isolation because it is small and specialized. The spectrum is expensive because the physics and parity constraints multiply that cheap primitive by exact support wavelengths, Fourier components, active layers, and doubling steps.

In [ ]:
MEASURED_DOUBLING_MS = 8_347.130
ratio = total_ms / MEASURED_DOUBLING_MS
print(f"estimated matrix primitive worker time: {total_ms:.3f} ms")
print(f"measured doubling worker time:          {MEASURED_DOUBLING_MS:.3f} ms")
print(f"estimate / measured:                    {ratio:.3f}")

## What This Says About the Wall

The per-spectrum wall is not a single slow line. It is a multiplicative structure:

```text
T_spectrum ~= exact_support_wavelengths
           * Fourier_terms_per_wavelength
           * active_layer_work_per_Fourier_term
           * doubling_steps_per_active_layer
           * small_matrix_kernel_cost
```

The first-order optimization boundary is therefore clear. A material speedup has to reduce one of the multiplicative factors or replace a core primitive without breaking the exact DISAMAR-parity path. Small improvements to nominal-grid integration or result collection cannot move the wall much because those steps happen after the expensive exact support solves have already been paid for.